In [2]:
import nba_api
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd
import numpy as np

In [5]:
seasons = ["2025-26", "2024-25", "2023-24", "2022-23", "2021-22", "2020-21", "2019-20", "2018-19", "2017-18", "2016-17", "2015-16", "2014-15", "2013-14", "2012-13", "2011-12", "2010-11", "2009-10", "2008-09", "2007-08", "2006-07", "2005-06", "2004-05", "2003-04", "2002-03", "2001-02", "2000-01"]

import time
from requests.exceptions import RequestException
from nba_api.stats.library.parameters import SeasonAll

dfs = []

for season in seasons:
    try:
        stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            per_mode_detailed="PerGame"
        )
        df = stats.get_data_frames()[0]
        df["season"] = season
        dfs.append(df)
    except (ValueError, RequestException, Exception) as e:
        print(f"Failed to fetch data for season {season}: {e}")
    time.sleep(1)  # Add delay to avoid rate limiting

if dfs:
    all_stats = pd.concat(dfs, ignore_index=True)
    print(all_stats.shape)
else:
    print("No data fetched.")

(12798, 68)


In [11]:
print("columns:", all_stats.columns)
columns_to_keep = ['PLAYER_ID', 'PLAYER_NAME', 'AGE',  'season', 'GP',  'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'DD2', 'TD3']

player_stats = all_stats[columns_to_keep]
print(player_stats.head())
player_stats['season'] = player_stats['season'].str[:4].astype(int)
player_stats.head()
player_stats.to_csv("player_stats.csv", index=False)


columns: Index(['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS',
       'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK',
       'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK',
       'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK',
       'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK',
       'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK',
       'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK',
       'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT',
       'season'],
      dtype='str')
   PLAYER_ID    PLAYER_NAME   AGE   season  GP  ...   PF   PTS  PLUS_MINUS  DD2  TD3
0    1630639    A.J. Lawson  25.0  2025-26 

In [ ]:
# MODEL_READY_FEATURES: engineer lagged and rate-based features for salary forecasting
player_stats_enhanced = pd.read_csv("player_stats.csv").copy()
player_stats_enhanced = player_stats_enhanced.rename(columns={"PLAYER_NAME": "Player", "season": "Season"})
player_stats_enhanced = player_stats_enhanced.sort_values(["PLAYER_ID", "Season"]).reset_index(drop=True)

per_36_stats = ["PTS", "REB", "AST", "STL", "BLK", "TOV"]
for stat in per_36_stats:
    player_stats_enhanced[f"{stat}_per36"] = np.where(
        player_stats_enhanced["MIN"] > 0,
        player_stats_enhanced[stat] / player_stats_enhanced["MIN"] * 36,
        0,
    )

player_stats_enhanced["AST_TOV_ratio"] = np.where(
    player_stats_enhanced["TOV"] > 0,
    player_stats_enhanced["AST"] / player_stats_enhanced["TOV"],
    player_stats_enhanced["AST"],
)
player_stats_enhanced["FG3A_rate"] = np.where(
    player_stats_enhanced["FGA"] > 0,
    player_stats_enhanced["FG3A"] / player_stats_enhanced["FGA"],
    0,
)
player_stats_enhanced["FTA_rate"] = np.where(
    player_stats_enhanced["FGA"] > 0,
    player_stats_enhanced["FTA"] / player_stats_enhanced["FGA"],
    0,
)
player_stats_enhanced["TS_proxy"] = np.where(
    (player_stats_enhanced["FGA"] + 0.44 * player_stats_enhanced["FTA"]) > 0,
    player_stats_enhanced["PTS"] / (2 * (player_stats_enhanced["FGA"] + 0.44 * player_stats_enhanced["FTA"])),
    0,
)
player_stats_enhanced["seasons_played"] = player_stats_enhanced.groupby("PLAYER_ID").cumcount()

lag_features = [
    "GP", "MIN", "PTS", "REB", "AST", "STL", "BLK", "TOV", "FG_PCT", "FG3_PCT",
    "FT_PCT", "PLUS_MINUS", "DD2", "TD3", "PTS_per36", "REB_per36", "AST_per36",
    "STL_per36", "BLK_per36", "TOV_per36", "AST_TOV_ratio", "FG3A_rate", "FTA_rate", "TS_proxy",
]
for feature in lag_features:
    player_stats_enhanced[f"{feature}_lag1"] = player_stats_enhanced.groupby("PLAYER_ID")[feature].shift(1)
    player_stats_enhanced[f"{feature}_rolling3"] = (
        player_stats_enhanced.groupby("PLAYER_ID")[feature]
        .transform(lambda values: values.shift(1).rolling(3, min_periods=1).mean())
    )

ambiguous_name_seasons = (
    player_stats_enhanced.groupby(["Player", "Season"])["PLAYER_ID"]
    .nunique()
    .reset_index(name="player_ids")
    .query("player_ids > 1")
    .sort_values(["Season", "Player"])
)

print("Engineered dataset shape:", player_stats_enhanced.shape)
print("Ambiguous player-season combinations:", len(ambiguous_name_seasons))
ambiguous_name_seasons.head(10)


In [ ]:
# MODEL_READY_FEATURES: save a cleaned modeling dataset that avoids ambiguous name matches
player_stats_model_ready = player_stats_enhanced.merge(
    ambiguous_name_seasons[["Player", "Season"]].assign(_ambiguous_name=1),
    on=["Player", "Season"],
    how="left",
)
player_stats_model_ready = player_stats_model_ready[player_stats_model_ready["_ambiguous_name"].isna()].drop(columns=["_ambiguous_name"])

print("Original stat rows:", len(player_stats_enhanced))
print("Rows kept for modeling:", len(player_stats_model_ready))
print("Rows removed because of ambiguous player names:", len(player_stats_enhanced) - len(player_stats_model_ready))

player_stats_model_ready.to_csv("player_stats_model_ready.csv", index=False)
player_stats_model_ready.head()
